In [7]:
#adding data from total-natural-resource-share to merged_renewable_gdp file

import pandas as pd

df_main  = pd.read_csv('merged_renewable_gdp.csv')
df_rents = pd.read_csv('total-natural-resource-rents.csv', skiprows=4)

# 2. Drop trailing empty column from total-natural-resource-rents file
df_rents = df_rents.drop(columns=['Unnamed: 70'], errors='ignore')

# 3. Remove regional/income aggregates like continents ("Africa"), "sub-saharan", etc
WB_AGGREGATES = {
    'AFE','AFW','ARB','CEB','EAR','EAS','EAP','ECS','ECA','EMU','EUU',
    'FCS','HIC','HPC','IBD','IBT','IDA','IDB','IDX','LAC','LDC','LIC',
    'LMC','LTE','MIC','MNA','NAC','OED','OSS','PRE','PSS','PST','SAS',
    'SSA','SSF','SST','TEA','TEC','TLA','TMN','TSA','TSS','UMC','WLD'
}
df_rents = df_rents[~df_rents['Country Code'].isin(WB_AGGREGATES)].copy()

# 4. Reshape from wide to long
id_cols = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
year_cols = [c for c in df_rents.columns if c not in id_cols]

df_rents_long = df_rents.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name='Year',
    value_name='natural_resource_rents_pct_gdp'
)
df_rents_long['Year'] = df_rents_long['Year'].astype(int)
df_rents_long = df_rents_long[['Country Code', 'Year', 'natural_resource_rents_pct_gdp']]

# 5. Restrict to years from 1965 onwards
year_min, year_max = df_main['Year'].min(), df_main['Year'].max()
df_rents_long = df_rents_long[df_rents_long['Year'].between(year_min, year_max)]

# 6. Left join on iso3 + Year
df_merged = df_main.merge(
    df_rents_long,
    how='left',
    left_on=['iso3', 'Year'],
    right_on=['Country Code', 'Year']
).drop(columns=['Country Code'])

# 7. Flag missing rents rows
df_merged['rents_missing'] = df_merged['natural_resource_rents_pct_gdp'].isna()

df_merged.to_csv('merged_renewable_gdp_rents.csv', index=False)
print("Done. Shape:", df_merged.shape)
print(df_merged.head())

Done. Shape: (4459, 10)
  country_owid country_wb iso3  Year  renewables_pct  GDP_per_capita  \
0      Algeria    Algeria  DZA  1965        4.487350      253.622060   
1      Algeria    Algeria  DZA  1966        3.312554      241.448970   
2      Algeria    Algeria  DZA  1967        4.042350      261.792442   
3      Algeria    Algeria  DZA  1968        5.170891      292.436036   
4      Algeria    Algeria  DZA  1969        2.995647      315.914656   

   gdp_missing missing_reason  natural_resource_rents_pct_gdp  rents_missing  
0        False            NaN                             NaN           True  
1        False            NaN                             NaN           True  
2        False            NaN                             NaN           True  
3        False            NaN                             NaN           True  
4        False            NaN                             NaN           True  


In [9]:
import pandas as pd
import statsmodels.formula.api as smf
 
# Replace this with your merged dataset from Day 4:
# renewable share + GDP per capita + at least one policy/economic control.
df = pd.read_csv("merged_renewable_gdp_rents.csv")
df = df.dropna(subset=["renewables_pct", "GDP_per_capita"])
 
# Simple OLS
model = smf.ols("renewables_pct ~ GDP_per_capita", data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         renewables_pct   R-squared:                       0.061
Model:                            OLS   Adj. R-squared:                  0.061
Method:                 Least Squares   F-statistic:                     273.2
Date:                Tue, 02 Jun 2026   Prob (F-statistic):           1.71e-59
Time:                        19:02:32   Log-Likelihood:                -16909.
No. Observations:                4190   AIC:                         3.382e+04
Df Residuals:                    4188   BIC:                         3.384e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          8.3446      0.266     31.

**Lets see if adding "natural_resource_rents_pct_gdp" as another variable effects our regression results.**

In [8]:
import pandas as pd
import statsmodels.formula.api as smf
 
# Replace this with your merged dataset from Day 4:
# renewable share + GDP per capita + at least one policy/economic control.
df = pd.read_csv("merged_renewable_gdp_rents.csv")
df = df.dropna(subset=["renewables_pct", "GDP_per_capita", "natural_resource_rents_pct_gdp"])
 
# Simple OLS
model = smf.ols("renewables_pct ~ GDP_per_capita + natural_resource_rents_pct_gdp", data=df).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:         renewables_pct   R-squared:                       0.098
Model:                            OLS   Adj. R-squared:                  0.097
Method:                 Least Squares   F-statistic:                     193.9
Date:                Tue, 02 Jun 2026   Prob (F-statistic):           1.06e-80
Time:                        19:01:49   Log-Likelihood:                -14417.
No. Observations:                3591   AIC:                         2.884e+04
Df Residuals:                    3588   BIC:                         2.886e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

In [10]:
import numpy as np
 
df["log_renew"]  = np.log(df["renewables_pct"].replace(0, np.nan))
df["log_gdp"]    = np.log(df["GDP_per_capita"])
df["log_natural_resource_rents"]    = np.log(df["natural_resource_rents_pct_gdp"].replace(0, np.nan))
 
loglog = smf.ols("log_renew ~ log_gdp + log_natural_resource_rents",
                 data=df.dropna(subset=["log_renew","log_gdp","log_natural_resource_rents"])).fit()
print(loglog.summary())


                            OLS Regression Results                            
Dep. Variable:              log_renew   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     7.669
Date:                Tue, 02 Jun 2026   Prob (F-statistic):           0.000476
Time:                        19:13:46   Log-Likelihood:                -6831.5
No. Observations:                3265   AIC:                         1.367e+04
Df Residuals:                    3262   BIC:                         1.369e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

In [14]:
robust = smf.ols("log_renew ~ log_gdp + log_natural_resource_rents",
                 data=df.dropna(subset=["log_renew","log_gdp","log_natural_resource_rents"]))\
            .fit(cov_type="HC3")
print(robust.summary())
 
# Country fixed effects — add C(Entity) for a categorical
fe = smf.ols("log_renew ~ log_gdp + log_natural_resource_rents + C(country_owid)",
             data=df.dropna(subset=["log_renew","log_gdp","log_natural_resource_rents"]))\
        .fit(cov_type="HC3")
# Show only the non-FE coefficients to keep the table readable
print(fe.summary().tables[1].as_text()[:2000])



                            OLS Regression Results                            
Dep. Variable:              log_renew   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     3.273
Date:                Wed, 03 Jun 2026   Prob (F-statistic):             0.0380
Time:                        09:45:30   Log-Likelihood:                -6831.5
No. Observations:                3265   AIC:                         1.367e+04
Df Residuals:                    3262   BIC:                         1.369e+04
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           